# 02 - Data Cleaning, Standardization & Storage

## Road Safety Data Pipeline

### Objectives
1. Menghapus data duplikat.
2. Melakukan standardisasi data kategorikal.
3. Mengonversi format tanggal.
4. Melakukan validasi kualitas data.
5. Menyimpan dataset bersih dalam format CSV dan Parquet.
6. Membandingkan ukuran penyimpanan kedua format.

## 2 — Import library dan Spark Session

In [4]:
import os

os.environ["HADOOP_HOME"] = "C:/hadoop"
os.environ["hadoop.home.dir"] = "C:/hadoop"
os.environ["PATH"] = "C:/hadoop/bin;" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RoadSafetyDataPipeline")
    .master("local[*]")
    .config(
        "spark.driver.extraJavaOptions",
        "-Dhadoop.home.dir=C:/hadoop"
    )
    .config(
        "spark.executor.extraJavaOptions",
        "-Dhadoop.home.dir=C:/hadoop"
    )
    .getOrCreate()
)

In [5]:
jvm = spark.sparkContext._jvm

print("Java property:",
      jvm.java.lang.System.getProperty("hadoop.home.dir"))

print("Winutils:",
      jvm.org.apache.hadoop.util.Shell.getWinUtilsPath())

Java property: C:/hadoop
Winutils: C:\hadoop\bin\winutils.exe


In [6]:
import findspark
import pyspark

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    upper,
    to_timestamp,
    date_format,
    dayofweek,
    month,
    hour,
    sum as spark_sum
)

findspark.init()

spark = (
    SparkSession.builder
    .appName("RoadSafetyDataCleaning")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark Version:", spark.version)

Spark Version: 4.3.0


## 3 — Menentukan direktori

In [7]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "traffic_accidents.csv"

CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUTPUT_PATH = CLEANED_DIR / "traffic_accidents_clean.csv"
PARQUET_OUTPUT_PATH = CLEANED_DIR / "traffic_accidents_clean.parquet"

print("Project Root:", PROJECT_ROOT)
print("Raw Dataset:", RAW_DATA_PATH)
print("Output Directory:", CLEANED_DIR)

Project Root: d:\UMM\SEMESTER 7\Analisis Big Data\UAS\road-safety-data-pipeline
Raw Dataset: d:\UMM\SEMESTER 7\Analisis Big Data\UAS\road-safety-data-pipeline\data\raw\traffic_accidents.csv
Output Directory: d:\UMM\SEMESTER 7\Analisis Big Data\UAS\road-safety-data-pipeline\data\cleaned


## 4 — Membaca dataset mentah

In [8]:
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW_DATA_PATH))
)

initial_count = df_raw.count()

print("Jumlah data awal:", initial_count)
df_raw.show(5, truncate=False)

Jumlah data awal: 209306
+----------------------+----------------------+-----------------+----------------------+----------------+---------------+------------------+--------------------+-----------+--------------------------------+----------------------+-------------+-----------------------------------+---------+------------------------+--------------+--------------+-----------------------+---------------------------+-----------------------------+----------------------+----------+-----------------+-----------+
|crash_date            |traffic_control_device|weather_condition|lighting_condition    |first_crash_type|trafficway_type|alignment         |roadway_surface_cond|road_defect|crash_type                      |intersection_related_i|damage       |prim_contributory_cause            |num_units|most_severe_injury      |injuries_total|injuries_fatal|injuries_incapacitating|injuries_non_incapacitating|injuries_reported_not_evident|injuries_no_indication|crash_hour|crash_day_of_week|crash_

## 5 — Menghapus duplikasi

In [9]:
df_clean = df_raw.dropDuplicates()

after_deduplication = df_clean.count()

removed_duplicates = initial_count - after_deduplication

print("Jumlah data awal       :", initial_count)
print("Duplikasi dihapus      :", removed_duplicates)
print("Jumlah data setelahnya :", after_deduplication)

Jumlah data awal       : 209306
Duplikasi dihapus      : 31
Jumlah data setelahnya : 209275


## 6 — Standardisasi kolom kategorikal

In [10]:
categorical_columns = [
    "traffic_control_device",
    "weather_condition",
    "lighting_condition",
    "first_crash_type",
    "trafficway_type",
    "alignment",
    "roadway_surface_cond",
    "road_defect",
    "crash_type",
    "intersection_related_i",
    "damage",
    "prim_contributory_cause",
    "most_severe_injury"
]

for column_name in categorical_columns:
    df_clean = df_clean.withColumn(
        column_name,
        upper(trim(col(column_name)))
    )

print("Standardisasi kategori selesai.")

Standardisasi kategori selesai.


## 7 — Konversi format tanggal

In [11]:
df_clean = df_clean.withColumn(
    "crash_date",
    to_timestamp(
        col("crash_date"),
        "MM/dd/yyyy hh:mm:ss a"
    )
)

df_clean.printSchema()

root
 |-- crash_date: timestamp (nullable = true)
 |-- traffic_control_device: string (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- lighting_condition: string (nullable = true)
 |-- first_crash_type: string (nullable = true)
 |-- trafficway_type: string (nullable = true)
 |-- alignment: string (nullable = true)
 |-- roadway_surface_cond: string (nullable = true)
 |-- road_defect: string (nullable = true)
 |-- crash_type: string (nullable = true)
 |-- intersection_related_i: string (nullable = true)
 |-- damage: string (nullable = true)
 |-- prim_contributory_cause: string (nullable = true)
 |-- num_units: integer (nullable = true)
 |-- most_severe_injury: string (nullable = true)
 |-- injuries_total: double (nullable = true)
 |-- injuries_fatal: double (nullable = true)
 |-- injuries_incapacitating: double (nullable = true)
 |-- injuries_non_incapacitating: double (nullable = true)
 |-- injuries_reported_not_evident: double (nullable = true)
 |-- injuries_no_i

## 8 — Validasi tanggal

In [12]:
invalid_date_count = df_clean.filter(
    col("crash_date").isNull()
).count()

print("Jumlah tanggal tidak berhasil dikonversi:", invalid_date_count)

Jumlah tanggal tidak berhasil dikonversi: 0


## 9 — Validasi NULL dan nilai kosong

In [13]:
null_summary = df_clean.select([
    count(
        when(
            col(c).isNull() |
            (trim(col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df_clean.columns
])

null_summary.show(truncate=False)

+----------+----------------------+-----------------+------------------+----------------+---------------+---------+--------------------+-----------+----------+----------------------+------+-----------------------+---------+------------------+--------------+--------------+-----------------------+---------------------------+-----------------------------+----------------------+----------+-----------------+-----------+
|crash_date|traffic_control_device|weather_condition|lighting_condition|first_crash_type|trafficway_type|alignment|roadway_surface_cond|road_defect|crash_type|intersection_related_i|damage|prim_contributory_cause|num_units|most_severe_injury|injuries_total|injuries_fatal|injuries_incapacitating|injuries_non_incapacitating|injuries_reported_not_evident|injuries_no_indication|crash_hour|crash_day_of_week|crash_month|
+----------+----------------------+-----------------+------------------+----------------+---------------+---------+--------------------+-----------+----------+---

## 10 — Validasi rentang nilai

In [14]:
invalid_range = df_clean.filter(
    (col("num_units") < 1) |
    (~col("crash_hour").between(0, 23)) |
    (~col("crash_day_of_week").between(1, 7)) |
    (~col("crash_month").between(1, 12)) |
    (col("injuries_total") < 0) |
    (col("injuries_fatal") < 0) |
    (col("injuries_incapacitating") < 0) |
    (col("injuries_non_incapacitating") < 0) |
    (col("injuries_reported_not_evident") < 0) |
    (col("injuries_no_indication") < 0)
)

print("Jumlah baris dengan nilai di luar rentang:", invalid_range.count())

Jumlah baris dengan nilai di luar rentang: 0


## 11 — Validasi konsistensi jumlah cedera

In [15]:
injury_components = (
    col("injuries_fatal") +
    col("injuries_incapacitating") +
    col("injuries_non_incapacitating") +
    col("injuries_reported_not_evident")
)

invalid_injury_total = df_clean.filter(
    col("injuries_total") != injury_components
)

print(
    "Jumlah baris dengan total cedera tidak konsisten:",
    invalid_injury_total.count()
)

Jumlah baris dengan total cedera tidak konsisten: 0


## 12 — Validasi konsistensi informasi tanggal

In [16]:
invalid_datetime = df_clean.filter(
    (hour(col("crash_date")) != col("crash_hour")) |
    (dayofweek(col("crash_date")) != col("crash_day_of_week")) |
    (month(col("crash_date")) != col("crash_month"))
)

print(
    "Jumlah baris dengan informasi waktu tidak konsisten:",
    invalid_datetime.count()
)

Jumlah baris dengan informasi waktu tidak konsisten: 0


## 13 — Validasi duplikasi setelah cleaning

In [17]:
final_count = df_clean.count()

remaining_duplicates = (
    final_count - df_clean.dropDuplicates().count()
)

print("Jumlah data akhir:", final_count)
print("Duplikasi tersisa:", remaining_duplicates)

Jumlah data akhir: 209275
Duplikasi tersisa: 0


## 14 — Ringkasan hasil cleaning

In [18]:
print("=" * 55)
print("DATA CLEANING & VALIDATION SUMMARY")
print("=" * 55)

print(f"Data awal                  : {initial_count:,}")
print(f"Duplikasi dihapus          : {removed_duplicates:,}")
print(f"Data setelah cleaning      : {final_count:,}")
print(f"Duplikasi tersisa          : {remaining_duplicates:,}")
print(f"Tanggal gagal dikonversi   : {invalid_date_count:,}")
print(f"Nilai di luar rentang      : {invalid_range.count():,}")
print(f"Total cedera tidak konsisten: {invalid_injury_total.count():,}")
print(f"Waktu tidak konsisten      : {invalid_datetime.count():,}")

print("=" * 55)

DATA CLEANING & VALIDATION SUMMARY
Data awal                  : 209,306
Duplikasi dihapus          : 31
Data setelah cleaning      : 209,275
Duplikasi tersisa          : 0
Tanggal gagal dikonversi   : 0
Nilai di luar rentang      : 0
Total cedera tidak konsisten: 0
Waktu tidak konsisten      : 0


## 15 — Menyimpan dataset dalam format Parquet

In [19]:
(
    df_clean
    .write
    .mode("overwrite")
    .parquet(str(PARQUET_OUTPUT_PATH))
)

print("Dataset Parquet berhasil disimpan.")
print("Lokasi:", PARQUET_OUTPUT_PATH)

Dataset Parquet berhasil disimpan.
Lokasi: d:\UMM\SEMESTER 7\Analisis Big Data\UAS\road-safety-data-pipeline\data\cleaned\traffic_accidents_clean.parquet


## 16 — Menyimpan dataset dalam format CSV

In [20]:
(
    df_clean
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(CSV_OUTPUT_PATH))
)

print("Dataset CSV berhasil disimpan.")
print("Lokasi:", CSV_OUTPUT_PATH)

Dataset CSV berhasil disimpan.
Lokasi: d:\UMM\SEMESTER 7\Analisis Big Data\UAS\road-safety-data-pipeline\data\cleaned\traffic_accidents_clean.csv


## 17 — Membaca ulang Parquet untuk memastikan penyimpanan berhasil

In [21]:
df_parquet = spark.read.parquet(str(PARQUET_OUTPUT_PATH))

print("Jumlah baris Parquet:", df_parquet.count())
print("Jumlah kolom Parquet:", len(df_parquet.columns))

df_parquet.printSchema()

Jumlah baris Parquet: 209275
Jumlah kolom Parquet: 24
root
 |-- crash_date: timestamp (nullable = true)
 |-- traffic_control_device: string (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- lighting_condition: string (nullable = true)
 |-- first_crash_type: string (nullable = true)
 |-- trafficway_type: string (nullable = true)
 |-- alignment: string (nullable = true)
 |-- roadway_surface_cond: string (nullable = true)
 |-- road_defect: string (nullable = true)
 |-- crash_type: string (nullable = true)
 |-- intersection_related_i: string (nullable = true)
 |-- damage: string (nullable = true)
 |-- prim_contributory_cause: string (nullable = true)
 |-- num_units: integer (nullable = true)
 |-- most_severe_injury: string (nullable = true)
 |-- injuries_total: double (nullable = true)
 |-- injuries_fatal: double (nullable = true)
 |-- injuries_incapacitating: double (nullable = true)
 |-- injuries_non_incapacitating: double (nullable = true)
 |-- injuries_reported_no

## 18 — Membandingkan ukuran penyimpanan

In [22]:
def get_directory_size(path):
    total_size = 0

    for file in Path(path).rglob("*"):
        if file.is_file():
            total_size += file.stat().st_size

    return total_size

csv_size = get_directory_size(CSV_OUTPUT_PATH)
parquet_size = get_directory_size(PARQUET_OUTPUT_PATH)

print(f"CSV Size     : {csv_size / (1024**2):.2f} MB")
print(f"Parquet Size : {parquet_size / (1024**2):.2f} MB")

if csv_size > 0:
    reduction = (1 - parquet_size / csv_size) * 100
    print(f"Ukuran Parquet dibanding CSV: {reduction:.2f}% lebih kecil")

CSV Size     : 50.70 MB
Parquet Size : 3.47 MB
Ukuran Parquet dibanding CSV: 93.16% lebih kecil
